In [2]:
# 7_synthetic_population.ipynb
#
# Expands the normalised UKHLS feature vectors onto the full SIPHER
# synthetic population by joining on pidp.
#
# Each synthetic person inherits the Z-scored feature profile of
# the UKHLS respondent they are based on, ready for regional
# clustering in a downstream step.
#
# Inputs:
#   sipher_optimized.pkl  — synthetic population (synthetic_zone, pidp)
#   normalized.pkl        — Z-scored UKHLS features per respondent (pidp + 27 features)
#
# Output:
#   synthetic_population.parquet — snappy-compressed; sorted by synthetic_zone
#                                   so per-zone reads use the filters= kwarg efficiently

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
SIPHER_PKL   = "../data/1_pickle_sipher/sipher_optimized.pkl"
NORM_PKL     = "../data/5_normalise_ukhls/normalized.pkl"
OUTPUT_FILE  = "../data/7_synthetic_population/synthetic_population.parquet"

# ── Load ──────────────────────────────────────────────────────────────────────
print("Loading data ...")
if not os.path.exists(SIPHER_PKL):
    raise FileNotFoundError(f"{SIPHER_PKL} not found — run 1_pickle_sipher.ipynb first.")
if not os.path.exists(NORM_PKL):
    raise FileNotFoundError(f"{NORM_PKL} not found — run 5_normalise_ukhls.ipynb first.")

df_sipher = pd.read_pickle(SIPHER_PKL)
df_norm   = pd.read_pickle(NORM_PKL)
print(f"Sipher:     {len(df_sipher):,} rows × {len(df_sipher.columns)} cols")
print(f"Normalised: {len(df_norm):,} rows × {len(df_norm.columns)} cols")

feature_cols = [c for c in df_norm.columns if c != 'pidp']
print(f"\nFeature columns ({len(feature_cols)}): {feature_cols}")

# Align pidp dtypes before merge
df_sipher['pidp'] = df_sipher['pidp'].astype('int64')
df_norm['pidp']   = df_norm['pidp'].astype('int64')

# ── Merge normalised features onto synthetic population ───────────────────────
print("\nExpanding normalised features onto synthetic population ...")
df_out = df_sipher.merge(df_norm[['pidp'] + feature_cols], on='pidp', how='left')

n_unmatched = df_out[feature_cols[0]].isna().sum()
if n_unmatched:
    print(f"  Warning: {n_unmatched:,} rows ({100*n_unmatched/len(df_out):.2f}%) could not be matched to a UKHLS respondent")
else:
    print(f"  All {len(df_out):,} rows matched successfully")

# Sort by zone so per-zone reads with filters= skip irrelevant row groups
df_out = df_out.sort_values('synthetic_zone').reset_index(drop=True)

# ── Save as single snappy-compressed Parquet ──────────────────────────────────
print(f"\nWriting Parquet to {OUTPUT_FILE} ...")
df_out.to_parquet(
    OUTPUT_FILE,
    engine='pyarrow',
    compression='snappy',
    index=False,
)

size_gb = Path(OUTPUT_FILE).stat().st_size / 1e9
print(f"\nSaved synthetic_population.parquet")
print(f"  Shape:   {df_out.shape[0]:,} rows × {df_out.shape[1]} columns")
print(f"  Disk:    {size_gb:.2f} GB  (snappy-compressed)")
print(f"\nTo read a single zone:")
print(f"  pd.read_parquet('{OUTPUT_FILE}', filters=[('synthetic_zone', '==', '<zone_id>')])")


Loading data ...
Sipher:     52,853,971 rows × 2 cols
Normalised: 47,354 rows × 28 cols

Feature columns (27): ['o_age_dv', 'o_hiqual_dv', 'o_payn_dv', 'o_jbnssec8_dv', 'o_fimngrs_dv', 'o_hhsize', 'o_nchild_dv', 'o_urban_dv', 'o_sf12mcs_dv', 'o_sf12pcs_dv', 'o_nbrsnci_dv', 'o_locserc', 'o_locserd', 'o_locsere', 'o_jbttwt', 'o_envhabit8', 'o_carmiles', 'o_englang_binary', 'o_work_at_home', 'o_drive_to_work', 'o_disability_mobility', 'o_disability_visual', 'o_disability_hearing', 'o_disability_learning', 'o_disability_mental_health', 'o_disability_dexterity', 'o_disability_memory']

Expanding normalised features onto synthetic population ...
  All 52,853,971 rows matched successfully

Writing Parquet to ../data/7_synthetic_population/synthetic_population.parquet ...

Saved synthetic_population.parquet
  Shape:   52,853,971 rows × 29 columns
  Disk:    0.82 GB  (snappy-compressed)

To read a single zone:
  pd.read_parquet('../data/7_synthetic_population/synthetic_population.parquet', filt